# Task C (Colab Edition)

SOC automation system prototype that reuses the inline Task B helpers. This version generates a triage queue, IOC summaries, and lightweight situational awareness visuals without importing repository modules.

In [ ]:
%%capture
!pip install -q pandas numpy joblib beautifulsoup4 lxml matplotlib

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from datetime import datetime, timedelta
from pathlib import Path
from typing import Any, Dict, List, Optional

import joblib
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
MODEL_PATH = ARTIFACT_DIR / "best_model.joblib"
FALLBACK_PATH = ARTIFACT_DIR / "ml" / "logistic_regression.joblib"
PROCESSED_TEST = PROJECT_ROOT / "data" / "processed" / "test.csv"

if not MODEL_PATH.exists() and not FALLBACK_PATH.exists():
    raise FileNotFoundError('Run Task A (Colab Edition) first so that artifacts/best_model.joblib exists.')
if not PROCESSED_TEST.exists():
    raise FileNotFoundError('Missing data/processed/test.csv. Execute Task A preprocessing before Task C.')

print('Artifacts ready:', ARTIFACT_DIR)
print('Test split located at:', PROCESSED_TEST)

In [ ]:
# -----------------------------------------------------------------------------
# Reuse Task B inline helpers (text cleaning, IOC extraction, inference)
# -----------------------------------------------------------------------------

import re


def strip_html(text: Optional[str]) -> str:
    if not text:
        return ''
    soup = BeautifulSoup(text, 'lxml')
    return soup.get_text(separator=' ').strip()


def normalize_text(text: Optional[str]) -> str:
    if not text:
        return ''
    text = text.lower()
    text = re.sub(r'https?://\S+', ' <URL> ', text)
    text = re.sub(r'[\w\.-]+@[\w\.-]+', ' <EMAIL> ', text)
    text = re.sub(r'\b\d+(?:\.\d+)?\b', ' <NUMBER> ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

IOC_PATTERNS = {
    'urls': re.compile(r'https?://[\w\-./?=&%]+', re.IGNORECASE),
    'ips': re.compile(r'(?:\d{1,3}\.){3}\d{1,3}'),
    'domains': re.compile(r'(?:[a-z0-9-]+\.)+[a-z]{2,}', re.IGNORECASE),
    'emails': re.compile(r'[\w\.-]+@[\w\.-]+\.[a-z]{2,}', re.IGNORECASE),
}


def extract_iocs(text: str) -> Dict[str, List[str]]:
    findings: Dict[str, List[str]] = {}
    for key, pattern in IOC_PATTERNS.items():
        matches = pattern.findall(text)
        if matches:
            uniq = []
            for item in matches:
                if item not in uniq:
                    uniq.append(item)
            findings[key] = uniq
    return findings


def summarize_iocs(iocs: Dict[str, List[str]]) -> Dict[str, int]:
    summary = {key: len(values) for key, values in iocs.items()}
    summary['total'] = sum(summary.values())
    return summary

_MODEL_CACHE = None


def _load_model():
    global _MODEL_CACHE
    if _MODEL_CACHE is not None:
        return _MODEL_CACHE
    if MODEL_PATH.exists():
        _MODEL_CACHE = joblib.load(MODEL_PATH)
    else:
        _MODEL_CACHE = joblib.load(FALLBACK_PATH)
    return _MODEL_CACHE


def classify_text(text: str) -> Dict[str, Any]:
    model = _load_model()
    cleaned = normalize_text(strip_html(text))
    if hasattr(model, 'predict_proba'):
        prob = model.predict_proba([cleaned])[:, 1]
    else:
        decision = model.decision_function([cleaned])
        prob = 1 / (1 + np.exp(-decision))
    probability = float(prob[0])
    label = int(probability >= 0.5)
    risk_level = _risk_level(probability)
    confidence = float(abs(probability - 0.5) * 2)
    return {
        'label': label,
        'score': probability,
        'risk_level': risk_level,
        'confidence': confidence,
    }


def _risk_level(probability: float) -> str:
    if probability >= 0.85:
        return 'high'
    if probability >= 0.65:
        return 'elevated'
    if probability >= 0.45:
        return 'moderate'
    return 'low'


def _recommend(label: int, risk: str) -> List[str]:
    if label == 1:
        steps = ['Quarantine message and block sender.', 'Open phishing investigation ticket.']
        if risk in {'high', 'elevated'}:
            steps.append('Trigger user password reset if credentials exposed.')
        return steps
    guidance = ['Log event for monitoring.']
    if risk in {'moderate', 'elevated'}:
        guidance.append('Perform manual review before releasing the message.')
    return guidance

In [ ]:
# -----------------------------------------------------------------------------
# Triage utilities
# -----------------------------------------------------------------------------

RISK_TO_SEVERITY = {
    'high': 'Critical',
    'elevated': 'High',
    'moderate': 'Medium',
    'low': 'Low',
}


def triage_message(subject: str, body: str, source: str, received_ts: datetime) -> Dict[str, Any]:
    combined = f"{subject}

{body}".strip()
    model_output = classify_text(combined)
    iocs = extract_iocs(combined)
    summary = summarize_iocs(iocs)
    probability = model_output['score']
    risk_level = model_output['risk_level']
    severity = RISK_TO_SEVERITY.get(risk_level, 'Low')
    priority_score = probability * 100 + summary.get('total', 0) * 5
    return {
        'timestamp': received_ts,
        'source': source,
        'subject': subject,
        'phishing_prob': probability,
        'label': model_output['label'],
        'risk_level': risk_level,
        'severity': severity,
        'priority_score': round(priority_score, 2),
        'confidence': model_output['confidence'],
        'ioc_summary': summary,
        'iocs': iocs,
        'recommendations': _recommend(model_output['label'], risk_level),
    }


def build_triage_queue(sample_size: int = 250, hours: int = 24) -> pd.DataFrame:
    df = pd.read_csv(PROCESSED_TEST)
    if len(df) == 0:
        raise ValueError('Test split is empty. Re-run Task A.')
    sample = df.sample(n=min(sample_size, len(df)), random_state=42)
    now = datetime.utcnow()
    rows: List[Dict[str, Any]] = []
    for row in sample.itertuples(index=False):
        subject = getattr(row, 'subject', '') or row.text[:80]
        body = getattr(row, 'text', '')
        source = getattr(row, 'source', 'email_gateway')
        delta = timedelta(hours=np.random.uniform(0, hours))
        event = triage_message(subject, body, source, now - delta)
        rows.append(event)
    triage_df = pd.DataFrame(rows)
    triage_df = triage_df.sort_values('timestamp').reset_index(drop=True)
    return triage_df

In [ ]:
triage_df = build_triage_queue(sample_size=200)
triage_df.head()

In [ ]:
# -----------------------------------------------------------------------------
# Visual summaries
# -----------------------------------------------------------------------------

plt.style.use('seaborn-v0_8-deep')
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
severity_counts = triage_df['severity'].value_counts().reindex(['Critical', 'High', 'Medium', 'Low']).fillna(0)
axes[0].bar(severity_counts.index, severity_counts.values, color=['#b71c1c', '#e65100', '#fdd835', '#43a047'])
axes[0].set_title('Alerts by severity')
axes[0].set_ylabel('Count')

triage_df = triage_df.sort_values('timestamp')
axes[1].plot(triage_df['timestamp'], triage_df['priority_score'], marker='o', linestyle='-')
axes[1].set_title('Priority score over time')
axes[1].set_ylabel('Priority score')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# -----------------------------------------------------------------------------
# Persist enriched triage output
# -----------------------------------------------------------------------------

reports_dir = PROJECT_ROOT / 'reports'
reports_dir.mkdir(parents=True, exist_ok=True)
triage_csv = reports_dir / 'soc_triage_queue.csv'
triage_json = reports_dir / 'soc_triage_queue.json'
triage_df.to_csv(triage_csv, index=False)
triage_df.to_json(triage_json, orient='records', indent=2, date_format='iso')
print('Saved triage CSV ->', triage_csv)
print('Saved triage JSON ->', triage_json)